In [2]:
!pip install -q langgraph langchain-huggingface langchain-core

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
import os
from google.colab import userdata

In [5]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')

In [6]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    provider="cerebras",   # plain chat, no structured output/tool calling -> cerebras is fine
    task="text-generation"
)
model = ChatHuggingFace(llm=llm_endpoint)

In [7]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [9]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    # NOTE: using a plain f-string here, not PromptTemplate, and that's intentional.
    # PromptTemplate exists to be a LangChain RUNNABLE -- something that plugs into
    # an LCEL pipe (prompt | model | parser), supports partial_variables, etc.
    # Inside a LangGraph NODE, there is no "|" pipe to join -- the node itself is
    # the unit of composition (nodes + edges do what "|" did for chains). A node
    # is meant to be "just a Python function," so building the prompt with an
    # f-string is the more idiomatic choice here, not a shortcut.
    # PromptTemplate would still work fine if used, it would just add Runnable
    # machinery (a PromptValue instead of a plain string) that this node has no
    # use for, since nothing downstream needs to chain off of it with "|".
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [10]:
# create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [11]:
# execute
initial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])

The Moon orbits the Earth at an average distance of about **384,400 km** (≈ 238,900 mi). Because its orbit is elliptical, the actual distance varies:

| Orbital point | Distance from Earth |
|---------------|---------------------|
| **Perigee** (closest) | ≈ 363,300 km (≈ 225,700 mi) |
| **Apogee** (farthest) | ≈ 405,500 km (≈ 251,900 mi) |

So, while the typical “average” figure quoted is ~384 km, the Moon can be roughly 20 km closer or farther depending on where it is in its orbit.


In [12]:
model.invoke('How far is moon from the earth?').content

'The Moon orbits the Earth at an average distance of **about\u202f384\u202f400\u202fkilometres (≈\u202f238\u202f900\u202fmiles)**.  \n\nBecause the Moon’s orbit is slightly elliptical, the distance varies:\n\n| Orbital position | Approximate distance from Earth |\n|------------------|---------------------------------|\n| Perigee (closest) | ~\u202f363\u202f300\u202fkm (≈\u202f225\u202f700\u202fmi) |\n| Apogee (farthest) | ~\u202f405\u202f500\u202fkm (≈\u202f251\u202f900\u202fmi) |\n\nSo the Moon is roughly 384\u202fkm away, give or take about ±\u202f21\u202fkm depending on where it is in its orbit.  \n\nFor perspective, **light takes about 1.28\u202fseconds** to travel from the Moon to Earth.'